In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'figure.figsize': (14, 6)})

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent

# ---- Configure dataset here ----
DATASET = "celebahq"  # or "celeba"
base = repo_root / "output" / "smile_classification" / DATASET
certify_dir = base / "certify"

# Mode tags match CertifyPaths.from_config:
#   {pixel|latent}_{manifold|isotropic}
MODES = ["pixel_isotropic", "pixel_manifold", "latent_isotropic", "latent_manifold"]
SIGMAS = ["sigma_0_25", "sigma_0_50", "sigma_0_75", "sigma_1_00"]

def sigma_val(s):
    return float(s.replace("sigma_", "").replace("_", "."))

def load_metrics(folder):
    """Load metrics.json from a certify experiment folder."""
    if not folder.exists():
        return None
    p = folder / "metrics.json"
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return None

def load_results_csv(folder):
    """Load per-sample results.csv."""
    p = folder / "results.csv"
    if p.exists():
        return pd.read_csv(p)
    return None

def mode_label(mode):
    """Pretty label: 'pixel_manifold' -> 'Pixel Manifold'."""
    return mode.replace('_', ' ').title()

# Discover what results exist
found = []
for mode in MODES:
    for sig in SIGMAS:
        p = certify_dir / mode / sig
        m = load_metrics(p)
        if m:
            found.append((mode, sig, sigma_val(sig)))

print(f"Dataset: {DATASET}")
print(f"Base: {base}")
print(f"Found {len(found)} experiment results:")
for mode, sig, sv in found:
    print(f"  {mode_label(mode):25s}  Ïƒ={sv}")

---
## 1. Grand Summary â€” All Experiments

In [ ]:
rows = []
for mode in MODES:
    for sig in SIGMAS:
        m = load_metrics(certify_dir / mode / sig)
        if m is None:
            continue
        rows.append({
            'mode': mode_label(mode),
            'space': mode.split('_')[0].title(),
            'smoothing': mode.split('_')[1].title(),
            'sigma': sigma_val(sig),
            'certified_acc': m.get('certified_accuracy', 0),
            'abstain_rate': m.get('abstain_rate', 0),
            'mean_radius': m.get('mean_radius', 0),
            'median_radius': m.get('median_radius', 0),
            'max_radius': m.get('max_radius', 0),
            'std_radius': m.get('std_radius', 0),
            'total': m.get('total_test_samples', 0),
            'certified_correct': m.get('certified_correct', 0),
            'smile_acc': m.get('class_smile_accuracy', 0),
            'no_smile_acc': m.get('class_no_smile_accuracy', 0),
            'smile_radius': m.get('class_smile_mean_radius', 0),
            'no_smile_radius': m.get('class_no_smile_mean_radius', 0),
        })

if rows:
    df = pd.DataFrame(rows)
    print("=" * 120)
    print(f"GRAND SUMMARY â€” {DATASET.upper()} SMILE CERTIFICATION")
    print("=" * 120)
    display(df[['mode', 'sigma', 'certified_acc', 'mean_radius', 'abstain_rate',
                'certified_correct', 'total', 'smile_acc', 'no_smile_acc']].round(4))
else:
    print("No results found. Copy server results to:", certify_dir)

---
## 2. Isotropic vs Manifold â€” Pixel Space

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    pixel = df[df['space'] == 'Pixel'].copy()
    
    if len(pixel) > 0:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for ax, metric, title in zip(axes,
            ['certified_acc', 'mean_radius', 'abstain_rate'],
            ['Certified Accuracy', 'Mean Certified Radius', 'Abstention Rate']):
            for sm in ['Isotropic', 'Manifold']:
                sub = pixel[pixel['smoothing'] == sm].sort_values('sigma')
                if len(sub) > 0:
                    ax.plot(sub['sigma'], sub[metric], 'o-', label=sm, linewidth=2, markersize=8)
            ax.set_xlabel('Ïƒ'); ax.set_ylabel(title); ax.set_title(title)
            ax.legend(); ax.grid(True, alpha=0.3)
        plt.suptitle(f'{DATASET.upper()} â€” Pixel Space: Isotropic vs Manifold', fontsize=14, y=1.02)
        plt.tight_layout(); plt.show()
    else:
        print("No pixel-space results found.")

---
## 3. Isotropic vs Manifold â€” Latent Space

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    latent = df[df['space'] == 'Latent'].copy()
    
    if len(latent) > 0:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for ax, metric, title in zip(axes,
            ['certified_acc', 'mean_radius', 'abstain_rate'],
            ['Certified Accuracy', 'Mean Certified Radius', 'Abstention Rate']):
            for sm in ['Isotropic', 'Manifold']:
                sub = latent[latent['smoothing'] == sm].sort_values('sigma')
                if len(sub) > 0:
                    ax.plot(sub['sigma'], sub[metric], 'o-', label=sm, linewidth=2, markersize=8)
            ax.set_xlabel('Ïƒ'); ax.set_ylabel(title); ax.set_title(title)
            ax.legend(); ax.grid(True, alpha=0.3)
        plt.suptitle(f'{DATASET.upper()} â€” Latent Space: Isotropic vs Manifold', fontsize=14, y=1.02)
        plt.tight_layout(); plt.show()
    else:
        print("No latent-space results found.")

---
## 4. Pixel vs Latent â€” All 4 Methods Compared

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = {'Pixel Isotropic': '#e41a1c', 'Pixel Manifold': '#377eb8',
              'Latent Isotropic': '#ff7f00', 'Latent Manifold': '#4daf4a'}
    
    for ax, metric, title in zip(axes,
        ['certified_acc', 'mean_radius', 'abstain_rate'],
        ['Certified Accuracy', 'Mean Certified Radius', 'Abstention Rate']):
        for mode_name in colors:
            sub = df[df['mode'] == mode_name].sort_values('sigma')
            if len(sub) > 0:
                ax.plot(sub['sigma'], sub[metric], 'o-', label=mode_name,
                        color=colors[mode_name], linewidth=2, markersize=8)
        ax.set_xlabel('Ïƒ'); ax.set_ylabel(title); ax.set_title(title)
        ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'{DATASET.upper()} â€” All 4 Methods Compared', fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()

---
## 5. Per-Class (Smile / No-Smile) Comparison

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors = {'Pixel Isotropic': '#e41a1c', 'Pixel Manifold': '#377eb8',
              'Latent Isotropic': '#ff7f00', 'Latent Manifold': '#4daf4a'}
    
    for ax, cls_metric, cls_name in zip(axes,
        ['smile_acc', 'no_smile_acc'], ['Smile', 'No-Smile']):
        for mode_name in colors:
            sub = df[df['mode'] == mode_name].sort_values('sigma')
            if len(sub) > 0:
                ax.plot(sub['sigma'], sub[cls_metric], 'o-', label=mode_name,
                        color=colors[mode_name], linewidth=2, markersize=8)
        ax.set_xlabel('Ïƒ'); ax.set_ylabel('Certified Accuracy')
        ax.set_title(f'{cls_name} Class â€” Certified Accuracy')
        ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'{DATASET.upper()} â€” Per-Class Certified Accuracy', fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()

---
## 6. Certification Radius Distribution (Per-Sample)

In [ ]:
# Load per-sample results for radius histograms at a chosen sigma
TARGET_SIGMA = "sigma_0_50"

csv_data = {}
for mode in MODES:
    df_csv = load_results_csv(certify_dir / mode / TARGET_SIGMA)
    if df_csv is not None:
        csv_data[mode_label(mode)] = df_csv

if csv_data:
    fig, axes = plt.subplots(1, len(csv_data), figsize=(5 * len(csv_data), 5), squeeze=False)
    axes = axes.flat
    
    for ax, (name, df_csv) in zip(axes, csv_data.items()):
        non_abstain = df_csv[df_csv['abstained'] == False]
        radii = non_abstain['radius'].values
        ax.hist(radii, bins=30, alpha=0.7, edgecolor='k')
        ax.axvline(radii.mean(), color='red', linestyle='--', label=f'mean={radii.mean():.3f}')
        ax.set_xlabel('Certified Radius'); ax.set_ylabel('Count')
        ax.set_title(name); ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'{DATASET.upper()} â€” Radius Distribution at Ïƒ={sigma_val(TARGET_SIGMA)}',
                 fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()
else:
    print(f"No per-sample results.csv found at Ïƒ={sigma_val(TARGET_SIGMA)}")

---
## 7. Certified Accuracy vs Radius Trade-off

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    colors = {'Pixel Isotropic': '#e41a1c', 'Pixel Manifold': '#377eb8',
              'Latent Isotropic': '#ff7f00', 'Latent Manifold': '#4daf4a'}
    
    fig, ax = plt.subplots(figsize=(10, 7))
    for mode_name in colors:
        sub = df[df['mode'] == mode_name].sort_values('sigma')
        if len(sub) > 0:
            ax.plot(sub['mean_radius'], sub['certified_acc'], 'o-', label=mode_name,
                    color=colors[mode_name], linewidth=2, markersize=10)
            # Annotate with sigma values
            for _, r in sub.iterrows():
                ax.annotate(f'Ïƒ={r["sigma"]}', (r['mean_radius'], r['certified_acc']),
                            textcoords='offset points', xytext=(6, 6), fontsize=8)
    
    ax.set_xlabel('Mean Certified Radius', fontsize=13)
    ax.set_ylabel('Certified Accuracy', fontsize=13)
    ax.set_title(f'{DATASET.upper()} â€” Accuracy vs Radius Trade-off', fontsize=14)
    ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

---
## 8. Heatmap â€” Certified Accuracy by Mode & Ïƒ

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    pivot = df.pivot_table(index='mode', columns='sigma', values='certified_acc', aggfunc='first')
    
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'Ïƒ={s}' for s in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            v = pivot.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.1%}', ha='center', va='center', fontsize=11,
                        fontweight='bold')
    
    plt.colorbar(im, label='Certified Accuracy')
    ax.set_title(f'{DATASET.upper()} â€” Certified Accuracy Heatmap', fontsize=14)
    plt.tight_layout(); plt.show()

---
## 9. Per-Sample Correct/Incorrect Breakdown

In [ ]:
# Stacked bar: correct / incorrect / abstained per mode at a chosen sigma
TARGET_SIGMA = "sigma_0_50"

bar_rows = []
for mode in MODES:
    m = load_metrics(certify_dir / mode / TARGET_SIGMA)
    if m:
        total = m['total_test_samples']
        correct = m['certified_correct']
        abstained = m['abstained_samples']
        wrong = total - correct - abstained
        bar_rows.append({
            'mode': mode_label(mode),
            'Correct': correct / total,
            'Wrong': wrong / total,
            'Abstained': abstained / total,
        })

if bar_rows:
    df_bar = pd.DataFrame(bar_rows).set_index('mode')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    df_bar.plot(kind='bar', stacked=True, ax=ax,
                color=['#4daf4a', '#e41a1c', '#999999'], alpha=0.8)
    ax.set_ylabel('Fraction'); ax.set_ylim(0, 1.05)
    ax.set_title(f'{DATASET.upper()} â€” Outcome Breakdown at Ïƒ={sigma_val(TARGET_SIGMA)}', fontsize=14)
    ax.legend(loc='upper right'); ax.grid(True, alpha=0.2, axis='y')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()
else:
    print(f"No results at Ïƒ={sigma_val(TARGET_SIGMA)}")

---
## 10. Per-Class Radius Comparison

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors = {'Pixel Isotropic': '#e41a1c', 'Pixel Manifold': '#377eb8',
              'Latent Isotropic': '#ff7f00', 'Latent Manifold': '#4daf4a'}
    
    for ax, cls_metric, cls_name in zip(axes,
        ['smile_radius', 'no_smile_radius'], ['Smile', 'No-Smile']):
        for mode_name in colors:
            sub = df[df['mode'] == mode_name].sort_values('sigma')
            if len(sub) > 0:
                ax.plot(sub['sigma'], sub[cls_metric], 'o-', label=mode_name,
                        color=colors[mode_name], linewidth=2, markersize=8)
        ax.set_xlabel('Ïƒ'); ax.set_ylabel('Mean Certified Radius')
        ax.set_title(f'{cls_name} Class â€” Mean Radius')
        ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'{DATASET.upper()} â€” Per-Class Certified Radius', fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()

---
## 11. Certified Volume Analysis â€” 4-Quantity Framework

| Qty | Formula | Radius | Meaning |
|-----|---------|--------|---------|
| 1 | $C_D \cdot r_{iso}^D$ | $r_{iso}$ | Classical RS volume (ambient D) |
| 2 | $C_k \cdot r_{iso}^k$ | $r_{iso}$ | Fair semantic baseline (dim k) |
| 3 | $C_k \cdot r_{iso}^k \cdot \sqrt{\det\Lambda}$ | $r_{iso}$ | Geometry-only manifold gain |
| 4 | $C_k \cdot r_{mani}^k \cdot \sqrt{\det\Lambda}$ | $r_{mani}$ | Actual manifold certificate |

- **Geometry factor** = $\frac{1}{2}\sum_i \log\lambda_i$ â€” r-independent, pure eigenvalue advantage

#### Hypothesis to test
- Qty 4 > Qty 3 means manifold smoothing also improves the radius (not just geometry)
- Qty 3 > Qty 2 means geometry alone helps (even at same radius)

In [ ]:
# 11a. All 4 Volume Quantities vs Ïƒ
volume_rows = []
for mode in MODES:
    for sig in SIGMAS:
        m = load_metrics(certify_dir / mode / sig)
        if m and "volume" in m:
            vol = m["volume"]
            volume_rows.append({
                'mode': mode_label(mode),
                'space': mode.split('_')[0].title(),
                'smoothing': mode.split('_')[1].title(),
                'sigma': sigma_val(sig),
                'mean_log_vol_iso_D': vol.get('mean_log_vol_iso_D'),        # Qty 1
                'mean_log_vol_iso_k': vol.get('mean_log_vol_iso_k'),        # Qty 2
                'mean_log_vol_mani_pred': vol.get('mean_log_vol_mani_pred'), # Qty 3
                'mean_log_vol_mani_actual': vol.get('mean_log_vol_mani_actual'),  # Qty 4
                'mean_geometry_factor': vol.get('mean_geometry_factor'),
                'mean_effective_rank': vol.get('mean_effective_rank'),
                'mean_condition_number': vol.get('mean_condition_number'),
                'k_pca': vol.get('k_pca'),
                'ambient_D': vol.get('ambient_D'),
            })

if volume_rows:
    df_vol = pd.DataFrame(volume_rows)
    
    # Only manifold modes have Qty 4
    mani_df = df_vol[df_vol['smoothing'] == 'Manifold']
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = {'Pixel Manifold': '#377eb8', 'Latent Manifold': '#4daf4a'}
    
    # Plot 1: All 4 quantities vs Ïƒ
    for mode_name, c in colors.items():
        sub = mani_df[mani_df['mode'] == mode_name].sort_values('sigma')
        if len(sub) == 0:
            continue
        # Qty 4: actual manifold (solid)
        axes[0].plot(sub['sigma'], sub['mean_log_vol_mani_actual'], 'o-',
                     label=f'{mode_name} Qty4 (actual)', color=c, linewidth=2, markersize=8)
        # Qty 3: predicted manifold (dashed)
        if sub['mean_log_vol_mani_pred'].notna().any():
            axes[0].plot(sub['sigma'], sub['mean_log_vol_mani_pred'], 's--',
                         label=f'{mode_name} Qty3 (pred)', color=c, linewidth=1.5, alpha=0.7)
        # Qty 2: iso at k (dotted)
        if sub['mean_log_vol_iso_k'].notna().any():
            axes[0].plot(sub['sigma'], sub['mean_log_vol_iso_k'], '^:',
                         label=f'{mode_name} Qty2 (iso,k)', color=c, linewidth=1, alpha=0.5)
    axes[0].set_xlabel('Ïƒ'); axes[0].set_ylabel('Mean Log Volume')
    axes[0].set_title('Certified Volumes vs Ïƒ (all at dim k)')
    axes[0].legend(fontsize=7); axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Geometry factor vs Ïƒ (r-independent)
    for mode_name, c in colors.items():
        sub = mani_df[mani_df['mode'] == mode_name].sort_values('sigma')
        if len(sub) > 0:
            axes[1].plot(sub['sigma'], sub['mean_geometry_factor'], 'o-',
                         label=mode_name, color=c, linewidth=2, markersize=8)
    axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[1].set_xlabel('Ïƒ'); axes[1].set_ylabel('0.5Â·Î£log(Î»_i)')
    axes[1].set_title('Geometry Factor (r-independent)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Effective rank vs Ïƒ
    for mode_name, c in colors.items():
        sub = mani_df[mani_df['mode'] == mode_name].sort_values('sigma')
        if len(sub) > 0:
            axes[2].plot(sub['sigma'], sub['mean_effective_rank'], 'o-',
                         label=mode_name, color=c, linewidth=2, markersize=8)
    axes[2].set_xlabel('Ïƒ'); axes[2].set_ylabel('Effective Rank')
    axes[2].set_title('Effective PCA Rank vs Ïƒ'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
    
    plt.suptitle(f'{DATASET.upper()} â€” 4-Quantity Volume Analysis', fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()
    
    # Summary table with all quantities
    cols = ['mode', 'sigma', 'k_pca', 'ambient_D',
            'mean_log_vol_iso_D', 'mean_log_vol_iso_k',
            'mean_log_vol_mani_pred', 'mean_log_vol_mani_actual',
            'mean_geometry_factor', 'mean_effective_rank']
    available_cols = [c for c in cols if c in df_vol.columns]
    print("\n4-Quantity Volume Summary:")
    display(df_vol[available_cols].round(2))
else:
    print("No volume data found. Run manifold experiments with updated code first.")

In [ ]:
# 11b. Eigenvalue Spectrum â€” from manifold runs at Ïƒ=0.50
for space in ['pixel', 'latent']:
    mode_tag = f"{space}_manifold"
    eigen_path = certify_dir / mode_tag / "sigma_0_50" / "eigenvalues.npz"
    
    if not eigen_path.exists():
        print(f"No eigenvalues.npz for {space} manifold at Ïƒ=0.50")
        continue
    
    data = np.load(eigen_path)
    all_evals = data.get('eigenvalues', np.array([]))
    if all_evals.ndim == 2:
        mean_evals = all_evals.mean(axis=0)
    else:
        mean_evals = all_evals
    
    k = len(mean_evals)
    sorted_evals = np.sort(mean_evals)[::-1]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Plot 1: Eigenvalue spectrum (log scale)
    axes[0].semilogy(range(k), sorted_evals, linewidth=2)
    axes[0].set_xlabel('PCA Component'); axes[0].set_ylabel('Eigenvalue (log)')
    axes[0].set_title(f'Mean Eigenvalue Spectrum (k={k})')
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Cumulative variance explained
    cumvar = np.cumsum(sorted_evals) / sorted_evals.sum()
    axes[1].plot(range(k), cumvar, linewidth=2)
    axes[1].axhline(0.95, color='red', linestyle='--', alpha=0.7, label='95%')
    axes[1].axhline(0.99, color='orange', linestyle='--', alpha=0.7, label='99%')
    axes[1].set_xlabel('Components'); axes[1].set_ylabel('Cumulative Variance')
    axes[1].set_title('Cumulative Variance Explained')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Geometry factor growth vs k
    log_det_halfs = [0.5 * np.sum(np.log(np.maximum(sorted_evals[:i+1], 1e-30))) for i in range(k)]
    axes[2].plot(range(1, k+1), log_det_halfs, linewidth=2, color='tab:red')
    axes[2].set_xlabel('k (PCA components)'); axes[2].set_ylabel('0.5 Ã— Î£ log(Î»_i)')
    axes[2].set_title('Geometry Factor vs k')
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle(f'{DATASET.upper()} â€” Eigenvalue Analysis: {space.title()} Space (Ïƒ=0.50)', fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()
    
    # Diagnostics
    from src.certify.randomized import eigenvalue_diagnostics
    diag = eigenvalue_diagnostics(mean_evals)
    print(f"  {space.title()}: k={diag.k}, eff_rank={diag.effective_rank:.1f}, "
          f"cond={diag.condition_number:.1f}, Î»_max={diag.lambda_max:.4f}, Î»_min={diag.lambda_min:.6f}")
    print()